# AI-Powered Plastic Sorting MVP - Part 1: Data Preparation

**Objective**: Download, explore, and prepare the plastic classification dataset

**Dataset**: [Plastic Type Recognition Dataset](https://www.kaggle.com/datasets/harshitkandoi7850/dataset-for-visual-plastic-type-recognition)

---

## 📦 Project Structure
```
plastic_sorting_mvp/
├── data/
│   ├── raw/              # Original dataset
│   ├── processed/        # Cleaned & balanced data
│   └── splits/           # Train/Val/Test splits
├── models/               # Saved models
├── notebooks/            # This notebook
└── app/                  # Streamlit app (Week 2)
```

## 1️⃣ Environment Setup

In [ ]:
# Install required packages
!pip install -q kaggle opendatasets
!pip install -q pillow matplotlib seaborn scikit-learn
!pip install -q tensorflow  # For model training

# Check GPU availability
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shutil
from collections import Counter
from PIL import Image
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 2️⃣ Download Dataset from Kaggle

**Before running this cell:**
1. Go to your Kaggle account settings: https://www.kaggle.com/settings
2. Scroll to "API" section
3. Click "Create New API Token" (downloads `kaggle.json`)
4. Upload the `kaggle.json` file to Colab using the file upload below

Then run the cells.

In [ ]:
# Upload kaggle.json
from google.colab import files
uploaded = files.upload()  # Upload your kaggle.json here

In [ ]:
# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d harshitkandoi7850/dataset-for-visual-plastic-type-recognition

# Unzip dataset
!unzip -q dataset-for-visual-plastic-type-recognition.zip -d plastic_dataset
print("✅ Dataset downloaded and extracted!")

In [ ]:
# Set paths
BASE_DIR = Path('plastic_dataset')
DATA_DIR = BASE_DIR / 'plastic_types'

# List all plastic categories
categories = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"📂 Found {len(categories)} categories:")
for cat in categories:
    print(f"  - {cat}")

## 3️⃣ Exploratory Data Analysis (EDA)

Let's understand what we're working with.

In [ ]:
# Count images per category
def count_images(data_dir):
    """Count images in each category folder."""
    image_counts = {}
    
    for category_path in data_dir.iterdir():
        if category_path.is_dir():
            # Count image files (jpg, jpeg, png)
            image_files = list(category_path.glob('*.jpg')) + \
                         list(category_path.glob('*.jpeg')) + \
                         list(category_path.glob('*.png'))
            image_counts[category_path.name] = len(image_files)
    
    return image_counts

# Get counts
image_counts = count_images(DATA_DIR)
total_images = sum(image_counts.values())

print(f"\n📊 Dataset Summary:")
print(f"Total Images: {total_images:,}")
print(f"\nImages per category:")
for cat, count in sorted(image_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_images) * 100
    print(f"  {cat:15s}: {count:4d} images ({percentage:5.2f}%)")

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

categories_sorted = sorted(image_counts.items(), key=lambda x: x[1], reverse=True)
cats, counts = zip(*categories_sorted)

bars = ax.bar(cats, counts, color='steelblue', edgecolor='black', alpha=0.7)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Plastic Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
ax.set_title('Class Distribution - Plastic Types Dataset', fontsize=14, fontweight='bold')
ax.axhline(y=np.mean(counts), color='red', linestyle='--', label=f'Mean: {int(np.mean(counts))}')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate imbalance ratio
max_count = max(counts)
min_count = min(counts)
imbalance_ratio = max_count / min_count
print(f"\n⚠️  Class Imbalance Ratio: {imbalance_ratio:.2f}x")
print(f"   (Max class has {imbalance_ratio:.2f}x more images than min class)")

In [ ]:
# Analyze image properties (dimensions, file sizes)
def analyze_image_properties(data_dir, sample_size=100):
    """Analyze dimensions and sizes of random sample of images."""
    
    widths, heights, file_sizes = [], [], []
    aspect_ratios = []
    corrupted = 0
    
    all_images = []
    for category_path in data_dir.iterdir():
        if category_path.is_dir():
            all_images.extend(list(category_path.glob('*.jpg')))
            all_images.extend(list(category_path.glob('*.jpeg')))
            all_images.extend(list(category_path.glob('*.png')))
    
    # Sample images
    np.random.seed(42)
    sample = np.random.choice(all_images, min(sample_size, len(all_images)), replace=False)
    
    for img_path in sample:
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / h)
                file_sizes.append(os.path.getsize(img_path) / 1024)  # KB
        except Exception as e:
            corrupted += 1
    
    return {
        'widths': widths,
        'heights': heights,
        'aspect_ratios': aspect_ratios,
        'file_sizes': file_sizes,
        'corrupted': corrupted
    }

# Analyze
print("🔍 Analyzing image properties (sampling 100 images)...")
props = analyze_image_properties(DATA_DIR, sample_size=100)

print(f"\n📐 Image Dimensions:")
print(f"  Width:  {np.mean(props['widths']):.0f} ± {np.std(props['widths']):.0f} px (min: {min(props['widths'])}, max: {max(props['widths'])})")
print(f"  Height: {np.mean(props['heights']):.0f} ± {np.std(props['heights']):.0f} px (min: {min(props['heights'])}, max: {max(props['heights'])})")
print(f"  Aspect Ratio: {np.mean(props['aspect_ratios']):.2f} ± {np.std(props['aspect_ratios']):.2f}")
print(f"\n💾 File Sizes:")
print(f"  Average: {np.mean(props['file_sizes']):.1f} KB")
print(f"  Range: {min(props['file_sizes']):.1f} - {max(props['file_sizes']):.1f} KB")
print(f"\n❌ Corrupted images found: {props['corrupted']}")

In [ ]:
# Visualize sample images from each category
def show_sample_images(data_dir, n_samples=3):
    """Display sample images from each category."""
    
    categories = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
    n_categories = len(categories)
    
    fig, axes = plt.subplots(n_categories, n_samples, figsize=(12, 2.5 * n_categories))
    
    for i, category in enumerate(categories):
        category_path = data_dir / category
        images = list(category_path.glob('*.jpg')) + list(category_path.glob('*.jpeg')) + list(category_path.glob('*.png'))
        
        # Random sample
        samples = np.random.choice(images, min(n_samples, len(images)), replace=False)
        
        for j, img_path in enumerate(samples):
            try:
                img = Image.open(img_path)
                axes[i, j].imshow(img)
                axes[i, j].axis('off')
                
                if j == 0:
                    axes[i, j].set_title(f'{category}\n{img.size[0]}x{img.size[1]}', 
                                        fontsize=10, fontweight='bold', loc='left')
                else:
                    axes[i, j].set_title(f'{img.size[0]}x{img.size[1]}', fontsize=9)
            except Exception as e:
                axes[i, j].text(0.5, 0.5, 'Error loading image', 
                              ha='center', va='center', fontsize=10)
                axes[i, j].axis('off')
    
    plt.suptitle('Sample Images from Each Plastic Type', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

# Show samples
np.random.seed(42)
show_sample_images(DATA_DIR, n_samples=3)

## 4️⃣ Data Quality Check

Check for corrupted images, duplicates, and data issues.

In [ ]:
# Check for corrupted/unreadable images
def find_corrupted_images(data_dir):
    """Find images that can't be opened."""
    
    corrupted = []
    
    for category_path in data_dir.iterdir():
        if category_path.is_dir():
            images = list(category_path.glob('*.jpg')) + \
                    list(category_path.glob('*.jpeg')) + \
                    list(category_path.glob('*.png'))
            
            for img_path in images:
                try:
                    with Image.open(img_path) as img:
                        img.verify()  # Verify it's a valid image
                except Exception as e:
                    corrupted.append((str(img_path), str(e)))
    
    return corrupted

print("🔍 Checking for corrupted images...")
corrupted_images = find_corrupted_images(DATA_DIR)

if corrupted_images:
    print(f"\n❌ Found {len(corrupted_images)} corrupted images:")
    for img_path, error in corrupted_images[:10]:  # Show first 10
        print(f"  {img_path}: {error}")
    if len(corrupted_images) > 10:
        print(f"  ... and {len(corrupted_images) - 10} more")
else:
    print("\n✅ No corrupted images found!")

## 5️⃣ Create Metadata DataFrame

Build a structured dataframe with all image paths and labels.

In [ ]:
# Create comprehensive metadata
def create_metadata_df(data_dir):
    """Create a dataframe with image paths, labels, and properties."""
    
    data = []
    
    for category_path in data_dir.iterdir():
        if category_path.is_dir():
            category = category_path.name
            
            images = list(category_path.glob('*.jpg')) + \
                    list(category_path.glob('*.jpeg')) + \
                    list(category_path.glob('*.png'))
            
            for img_path in images:
                try:
                    with Image.open(img_path) as img:
                        width, height = img.size
                        mode = img.mode
                    
                    file_size = os.path.getsize(img_path) / 1024  # KB
                    
                    data.append({
                        'filepath': str(img_path),
                        'filename': img_path.name,
                        'category': category,
                        'width': width,
                        'height': height,
                        'aspect_ratio': width / height,
                        'mode': mode,
                        'file_size_kb': file_size
                    })
                except Exception as e:
                    # Skip corrupted images
                    continue
    
    return pd.DataFrame(data)

# Create dataframe
print("📋 Creating metadata dataframe...")
df = create_metadata_df(DATA_DIR)

print(f"\n✅ Created dataframe with {len(df):,} images")
print(f"\nFirst few rows:")
df.head(10)

In [ ]:
# Save metadata
df.to_csv('plastic_metadata.csv', index=False)
print("💾 Saved metadata to 'plastic_metadata.csv'")

# Summary statistics
print("\n📊 Dataset Statistics:")
print(df.groupby('category').agg({
    'filepath': 'count',
    'width': ['mean', 'std'],
    'height': ['mean', 'std'],
    'file_size_kb': ['mean', 'std']
}).round(2))

## 6️⃣ Key Insights Summary

Based on the EDA, let's document findings.

In [ ]:
# Generate EDA summary report
eda_summary = {
    'total_images': len(df),
    'num_categories': df['category'].nunique(),
    'categories': df['category'].unique().tolist(),
    'class_distribution': df['category'].value_counts().to_dict(),
    'imbalance_ratio': df['category'].value_counts().max() / df['category'].value_counts().min(),
    'avg_image_size': {
        'width': float(df['width'].mean()),
        'height': float(df['height'].mean())
    },
    'corrupted_images': len(corrupted_images),
    'recommended_input_size': 224  # Standard for transfer learning
}

# Save summary
with open('eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print("📝 EDA Summary:")
print(json.dumps(eda_summary, indent=2))

print("\n✅ Part 1 Complete!")
print("\n📌 Next Steps:")
print("  1. Handle class imbalance (if ratio > 2x)")
print("  2. Create train/val/test splits")
print("  3. Build data pipeline with augmentation")
print("  4. Train transfer learning model")